# 1. Importação dos Dados e Bibliotecas.

Nesta etapa, serão importadas as bibliotecas Pandas e NumPy, juntamente com a base de clientes em sua versão bruta. Os dados serão carregados em um DataFrame para uma análise exploratória inicial, permitindo compreender sua estrutura, identificar inconsistências e definir as transformações necessárias para o processo de tratamento dos dados.

In [1]:
from pathlib import Path
import sys

RAIZ    = Path.cwd().parent
BRONZE  = RAIZ / "data" / "Bronze"
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

assert BRONZE.exists(), f"Pasta não encontrada: {BRONZE}. Abra o notebook a partir da pasta notebooks/."

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)
sys.path.append(str(RAIZ))         

## Importação das bibliotecas
import pandas as pd
import numpy as np

## Importando funções da pasta src.
from src.datas import converter_data, conferir_data

df_cliente = pd.read_csv(BRONZE / "clientes.csv")
original = df_cliente.copy()

# 2. Tratamento e Análise de Dados:

Nesta etapa, cada coluna da base será analisada individualmente, com o objetivo de identificar inconsistências, valores nulos, dados inválidos e possíveis problemas de formatação. A partir dessa análise, serão definidas e aplicadas as transformações necessárias para garantir a qualidade e a padronização dos dados.

## 2.1 Tratamento Coluna: id_cliente

Nesta etapa, foi analisada a existência de IDs duplicados. Por se tratar de uma coluna identificadora da dimensão (DIM), os IDs devem ser únicos. A existência de IDs duplicados pode gerar ambiguidades nos relacionamentos e, consequentemente, impactar os resultados dos relatórios. Sendo assim, foi realizada uma contagem dos IDs, com o objetivo de identificar aqueles que se repetem mais de uma vez na base.

In [2]:
# 1. Análise de Dados: Validando a quantidade de id_clientes que se repetem.
contagem_ids = df_cliente['id_cliente'].value_counts()
ids_repetidos = contagem_ids[contagem_ids > 1]
print(ids_repetidos)

# 2. Tratamento de Dados: Removendo linhas duplicadas dentro do Data Frame.
df_cliente = df_cliente.drop_duplicates()

# 3. Validação de dados: Verificação Pós Tratamento.
contagem_validacao = df_cliente['id_cliente'].value_counts()
resultado_duplicadas = contagem_validacao[contagem_validacao > 1]
print(resultado_duplicadas)
print(df_cliente.info())

id_cliente
443     2
448     2
614     2
675     2
809     2
1132    2
1150    2
1211    2
1283    2
1316    2
1469    2
1497    2
1637    2
1683    2
1700    2
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_cliente     2000 non-null   int64  
 1   nome           2000 non-null   str    
 2   idade          1975 non-null   float64
 3   cidade         2000 non-null   str    
 4   estado         1994 non-null   str    
 5   data_cadastro  2000 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 163.4 KB
None


## 2.2 Tratamento da Coluna nome

Nesta etapa, foram eliminados os espaços em branco no início e no final do nome completo, prevenindo divergências e inconsistências em futuras análises e filtros.

In [3]:
# 1. Tratamento de Dados: Remoção de espaços extras na coluna nome.
df_cliente['nome'] = df_cliente['nome'].str.strip()
print('Removendo espaços incorretos')

Removendo espaços incorretos


## 2.3 Tratamento Coluna Idade

Conversão do Tipo de Dado (Idade): A coluna idade foi convertida de float para Int64, garantindo que apenas valores inteiros sejam processados e evitando erros de tipagem.

Validação de Faixa Etária: Foi realizada uma verificação de consistência para identificar valores inválidos (idades menores ou iguais a 0, ou maiores e iguais a 120), assegurando a integridade da base de dados. Outro ponto importante, foi a questão de anular qualquer valor que seja <= 0 ou >= 120, trocando por um dado nulo.

In [4]:
# 1. Tratamento de Dados: transformando o type em Int
df_cliente['idade'] = df_cliente['idade'].astype('Int64')
print(f'Validação da Transformação {df_cliente.info()}')

# 2. Validação de Dados: Validando se existe idades divergentes.
df_invalidos = df_cliente[(df_cliente['idade'] <= 0) | (df_cliente['idade'] >= 120)]
print(df_invalidos)

# 3. Tratamento de Dados: Substituindo valores divergentes por NaN.
df_cliente['idade'] = df_cliente['idade'].mask(
    (df_cliente['idade'] <= 0) | (df_cliente['idade'] >= 120), 
    np.nan
)

# 4. Validação de Dados: Verificando as informações Pós Tratamento.
print(df_cliente[(df_cliente['idade'] <= 0) | (df_cliente['idade'] >= 120)])
print(df_cliente.info())

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     2000 non-null   int64
 1   nome           2000 non-null   str  
 2   idade          1975 non-null   Int64
 3   cidade         2000 non-null   str  
 4   estado         1994 non-null   str  
 5   data_cadastro  2000 non-null   str  
dtypes: Int64(1), int64(1), str(4)
memory usage: 165.4 KB
Validação da Transformação None
      id_cliente               nome  idade          cidade estado  \
99           100   Fernanda Pereira      0        Salvador     BA   
129          130   Fernanda Pereira      0       São Paulo     SP   
160          161      Juliana Costa    150  Rio de Janeiro     RJ   
200          201       Daniel Souza      0       São Paulo     SP   
357          358    Fernanda Santos    120        Campinas     SP   
522          523      Lucas Pereira     -5  Belo Horizonte     MG

## 2.4 Tratamento Coluna Cidade

Realizado a remoção de espaços excedentes no valores que estão na coluna cidade, a fim de seguir com a padronização de dados e evitar divergência em possíveis filtros. 

Padronização e Correção de Cidades: Nesta etapa, realizou-se a validação dos nomes informados na coluna cidade, identificando inconsistências e grafias incorretas. Para padronizar os dados de acordo com cada estado, os valores divergentes foram substituídos pelos nomes oficiais corretos, garantindo a integridade e a atualização do DataFrame.

In [5]:
# 1. Tratamento de Dados: Removendo os espaços extras de textos da coluna cidade
df_cliente['cidade'] = df_cliente['cidade'].str.strip()
print('Removendo espaços incorretos')

# 2. Exploração de Dados: Validando os dados que possui a coluna cidade
lista_cidade = df_cliente['cidade'].unique() 
print(f'Lista de cidades: {lista_cidade}')

correcao = {                   
    'SP': 'São Paulo',
    'sao paulo': 'São Paulo',
    'Sao Paulo': 'São Paulo',
    'rio de janeiro': 'Rio de Janeiro'
}
# 3.1 Tratamento de Dados: Aplicação da correção/padronização no data frame.
df_cliente['cidade'] = df_cliente['cidade'].replace(correcao)   

# 4. Validação de Dados: Validando informações pós tratamento.
lista_cidade = df_cliente['cidade'].unique()                 
print(f'Lista de cidades: {lista_cidade}')

Removendo espaços incorretos
Lista de cidades: <ArrowStringArray>
[      'Curitiba',      'Sao Paulo', 'Belo Horizonte',       'Salvador',
 'Rio de Janeiro',       'Campinas',      'São Paulo',      'sao paulo',
             'SP', 'rio de janeiro']
Length: 10, dtype: str
Lista de cidades: <ArrowStringArray>
[      'Curitiba',      'São Paulo', 'Belo Horizonte',       'Salvador',
 'Rio de Janeiro',       'Campinas']
Length: 6, dtype: str


## 2.5 Tratamento da Coluna Estado

Realizado a remoção de espaços excedentes no valores que estão na coluna cidade, a fim de seguir com a padronização de dados e evitar divergência em possíveis filtros.

Padronização e Correção de Estados: Foi realizada uma análise dos valores presentes na coluna estado para identificar inconsistências, grafias incorretas ou termos divergentes. Após a detecção dessas falhas, os dados foram devidamente corrigidos e padronizados, garantindo a atualização e a integridade do DataFrame.

In [6]:
# 1. Tratamento de Dados: Removendo espaços extras
df_cliente['estado'] = df_cliente['estado'].str.strip()
print('Removendo espaços incorretos')

# 2. Exploração de Dados: Validando os dados que possui a coluna estado
lista_estado = df_cliente['estado'].unique()
print(f'Lista de estados: {lista_estado}')

# 3. Tratamento de Dados: Criando um dicionário de correção, para aplicar no Data Frame.
cidade_para_estado = {
    # Região Norte
    'Rio Branco': 'AC',
    'Macapá': 'AP',
    'Manaus': 'AM',
    'Belém': 'PA',
    'Porto Velho': 'RO',
    'Boa Vista': 'RR',
    'Palmas': 'TO',

    # Região Nordeste
    'Maceió': 'AL',
    'Salvador': 'BA',
    'Fortaleza': 'CE',
    'São Luís': 'MA',
    'João Pessoa': 'PB',
    'Recife': 'PE',
    'Teresina': 'PI',
    'Natal': 'RN',
    'Aracaju': 'SE',

    # Região Centro-Oeste
    'Brasília': 'DF',
    'Goiânia': 'GO',
    'Cuiabá': 'MT',
    'Campo Grande': 'MS',

    # Região Sudeste
    'Belo Horizonte': 'MG',
    'Uberlândia': 'MG',
    'Rio de Janeiro': 'RJ',
    'Niterói': 'RJ',
    'São Paulo': 'SP',
    'Campinas': 'SP',
    'Santos': 'SP',
    'Ribeirão Preto': 'SP',
    'São José dos Campos': 'SP',
    'Vitória': 'ES',

    # Região Sul
    'Curitiba': 'PR',
    'Londrina': 'PR',
    'Maringá': 'PR',
    'Porto Alegre': 'RS',
    'Caxias do Sul': 'RS',
    'Florianópolis': 'SC',
    'Joinville': 'SC',
    'Blumenau': 'SC'
}

# 3.1 Tratamento de Dados: Atualiza a coluna estado com base no dicionário (cidades que não estiverem no dicionário virarão NaN)
df_cliente['estado'] = df_cliente['cidade'].map(cidade_para_estado)

# 4. Validação de Dados: Informa o resultado pós tratamento dos dados da coluna Estado.
print(df_cliente.info())
print(df_cliente['estado'].unique())


Removendo espaços incorretos
Lista de estados: <ArrowStringArray>
['PR', 'BA', 'MG', 'RJ', 'SP', nan, 'XX']
Length: 7, dtype: str
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     2000 non-null   int64
 1   nome           2000 non-null   str  
 2   idade          1955 non-null   Int64
 3   cidade         2000 non-null   str  
 4   estado         2000 non-null   str  
 5   data_cadastro  2000 non-null   str  
dtypes: Int64(1), int64(1), str(4)
memory usage: 165.2 KB
None
<ArrowStringArray>
['PR', 'SP', 'MG', 'BA', 'RJ']
Length: 5, dtype: str


## 2.6 Tratamento da Coluna data

Durante a análise do data frame, foi identificado que a coluna data, não estava configurado com type date, sendo assim, foi necessário realizado essa alteração, para transformar essa coluna em um tipo data, e após isso foi feito a validação e atualização do data frame.

In [7]:
# 1. Tratamento de Dados: Transformando o type da coluna data_cadastro em datetime.
df_cliente['data_cadastro'] = converter_data(df_cliente['data_cadastro'])

# 2. Validação de Dados: Validando dados Pós Tratamento:
## 2.1. O tipo é data de verdade? (esperado: datetime64)
print("Tipo:", df_cliente['data_cadastro'].dtype)

ok = conferir_data(df_cliente['data_cadastro'], original.loc[df_cliente.index, 'data_cadastro'])
print("Datas conferem com o original?", ok)

Tipo: datetime64[us]
Datas conferem com o original? True


# 3. Data Quality & Validação.

Nesta etapa, o DataFrame tratado é submetido a testes de validação antes de ser promovido para a pasta Staging

In [8]:
resultado_qualidade = {}

# Validação de IDs não nulos
resultado_qualidade['ID não nulo'] = (
    df_cliente['id_cliente'].isnull().sum() == 0
)

# Validação a fim de identificar se existem IDs duplicados
resultado_qualidade['ID único'] = (
    df_cliente['id_cliente'].duplicated().sum() == 0
)

# Validação se existem linhas duplicadas
resultado_qualidade['Sem duplicatas completas'] = (
    df_cliente.duplicated().sum() == 0
)

# Validação sobre se existe nomes que não foram preenchidos na coluna nome
resultado_qualidade['Nome preenchido'] = (
    df_cliente['nome'].notna().all()
    and
    df_cliente['nome'].str.strip().ne('').all()
)

# Validação a respeito da Idade
idades_invalidas = (
    (df_cliente['idade'] <= 0) |
    (df_cliente['idade'] >= 120)
).sum()

resultado_qualidade['Idades válidas'] = (
    idades_invalidas == 0
)

# Validação para identificar se existem algum dado que não está preenchido na coluna Cidade
resultado_qualidade['Cidade preenchida'] = (
    df_cliente['cidade'].notna().all()
    and
    df_cliente['cidade'].str.strip().ne('').all()
)

# Validação para identificar se os estados preenchidos na coluna estado, são validos:
estados_validos = [
    'AC', 'AL', 'AP', 'AM',
    'BA', 'CE', 'DF', 'ES',
    'GO', 'MA', 'MT', 'MS',
    'MG', 'PA', 'PB', 'PR',
    'PE', 'PI', 'RJ', 'RN',
    'RS', 'RO', 'RR', 'SC',
    'SP', 'SE', 'TO'
]

resultado_qualidade['Estados válidos'] = (
    df_cliente['estado'].isin(estados_validos).all()
)

# Validação para verificar se não houve erro na data de cadastro, onde possívelmente podem ter inserido datas errados, como por exemplo, datas futuras, onde por regra de negócio, não se pode cadastrar o cliente em uma data futura.
data_atual = pd.Timestamp.today().normalize()

resultado_qualidade['Sem datas futuras'] = (
    (df_cliente['data_cadastro'] <= data_atual).all()
)


# 4. Data Quality: Homologação e Envio para a Camada Staging (Preparação de Dados)

Nesta etapa, o console exibe o resultado das validações realizadas:

Aprovado (True em todos os testes): O DataFrame é liberado e enviado para a camada Staging (Preparação de Dados)
Reprovado (qualquer valor False): O processo é bloqueado, exigindo a correção dos dados antes de avançar.

In [9]:
df_qualidade = pd.DataFrame(
    list(resultado_qualidade.items()),
    columns=['Regra', 'Aprovado']
)

df_qualidade['Status'] = df_qualidade['Aprovado'].map({
    True: 'APROVADO',
    False: 'REPROVADO'
})

display(df_qualidade)


todos_testes_ok = df_qualidade['Aprovado'].all()
caminho_saida = (STAGING / 'dim_clientes.csv')

if todos_testes_ok:
    print("Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).")

    df_cliente.to_csv(
        caminho_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

else:
    print("Alerta: O arquivo foi impedido de subir para a camada Staging. Existem regras reprovadas.")
    raise ValueError("Falha na validação de qualidade dos dados. Processo interrompido.")

,Regra,Aprovado,Status
0,ID não nulo,True,APROVADO
1,ID único,True,APROVADO
2,Sem duplicatas completas,True,APROVADO
3,Nome preenchido,True,APROVADO
4,Idades válidas,True,APROVADO
5,Cidade preenchida,True,APROVADO
6,Estados válidos,True,APROVADO
7,Sem datas futuras,True,APROVADO


Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging (Preparação de Dados).
